# scRareBench | High-Level scDisInFact | Dataset 0 — BMMC (GSE194122 curated)

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This Google Colab notebook runs **scDisInFact** on scRareBench Dataset 0 (`gse194122`) through the generic **MethodSpec → benchmark_method** API.

## Important Dataset 0 adaptation

scDisInFact is designed for **multi-batch, multi-condition** data. Dataset 0 defines `BATCH` and the benchmark label `celltype`, but does **not** define an independent biological condition covariate.

Using `celltype` as a scDisInFact condition would leak the benchmark ground truth into the method. This notebook therefore adds one constant method-only condition (`baseline`) to all cells and benchmarks only the **shared biological factor `mu_c`**. This is the batch-integration use of scDisInFact, not a full multi-condition perturbation benchmark.

### Contract
- Dataset: selector `0` / GSE194122, 89,199 curated cells
- Batch: `BATCH`
- Benchmark label: `celltype` (evaluation only; never passed to scDisInFact)
- Method-only condition: constant `baseline`
- Features: 2,000 batch-aware Seurat-v3 HVGs from raw counts
- Latent dimensions: shared `8`, dummy-condition-specific `4`
- Evaluated representation: shared factor `mu_c` only
- Reconstruction loss: `NB`
- Training epochs: `100`
- Method seeds: `42, 123, 2026`
- Benchmark seed: fixed at `42`

The official scDisInFact preprocessing normalizes, log-transforms, and standardizes the input inside `create_scdisinfact_dataset()`, so this notebook passes raw counts after HVG selection.

> scDisInFact densifies the input internally. Use **GPU + High-RAM Colab**. For a smoke test, temporarily change `METHOD_SEEDS` to `[42]`.

## 1. Install scRareBench and pinned scDisInFact

scDisInFact is pinned to upstream commit `9466ecf257d923f879cb39712c530d24826d88bc` for reproducibility. The package reports version `0.1.0`.


In [ ]:
import subprocess, sys

SCRAREBENCH_GITHUB = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
SCDISINFACT_COMMIT = "9466ecf257d923f879cb39712c530d24826d88bc"
SCDISINFACT_GITHUB = "git+https://github.com/ZhangLabGT/scDisInFact.git@" + SCDISINFACT_COMMIT

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCRAREBENCH_GITHUB])

METHOD_RUNTIME_DEPENDENCIES = (
    "torch>=1.11.0",
    "scikit-learn>=0.22.1",
    "matplotlib>=3.5.2",
    "scikit-misc>=0.3",
)

from scrarebench.runtime import print_install_report, setup_runtime
install_report = setup_runtime(
    extra_requirements=METHOD_RUNTIME_DEPENDENCIES,
    extra_imports=("torch", "sklearn", "matplotlib", "skmisc"),
    quiet=False,
)
print_install_report(install_report)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCDISINFACT_GITHUB])

import scDisInFact
from scDisInFact import create_scdisinfact_dataset, scdisinfact
print("scDisInFact version:", getattr(scDisInFact, "__version__", "installed"))


In [ ]:
# OPTIONAL runtime diagnostics only.
# import torch, numpy as np, scanpy as sc, sklearn
# print("NumPy:", np.__version__)
# print("scanpy:", sc.__version__)
# print("scikit-learn:", sklearn.__version__)
# print("torch:", torch.__version__)
# print("CUDA available:", torch.cuda.is_available())
# if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))


## 2. Load curated Dataset 0


In [ ]:
from scrarebench import load_dataset, dataset_info

adata = load_dataset(0)
info = dataset_info(adata)
print(adata)
print(info)

assert adata.n_obs == 89_199
assert info["label_key"] == "celltype"
assert info["batch_key"] == "BATCH"
assert adata.obs_names.is_unique


## 3. Reproducibility and method configuration

`Ks=[8,4]` means an 8-dimensional shared factor plus one 4-dimensional condition-specific factor. Because Dataset 0 has no independent condition covariate, that condition has a single constant level and only `mu_c` is evaluated.


In [ ]:
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "n_hvg": 2000,
    "hvg_span": 0.3,
    "condition_column": "_scdisinfact_condition",
    "condition_value": "baseline",
    "Ks": [8, 4],
    "batch_size": 64,
    "interval": 10,
    "lr": 5e-4,
    "reg_mmd_comm": 1e-4,
    "reg_mmd_diff": 1e-4,
    "reg_gl": 1.0,
    "reg_class": 1.0,
    "reg_kl_comm": 1e-5,
    "reg_kl_diff": 1e-2,
    "enc_injection": False,
    "nepochs": 100,
    "recon_loss": "NB",
}

OUTPUT_DIR = "/content/scrarebench_scdisinfact_dataset0_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_scdisinfact_method_runs"


## 4. User-owned scDisInFact runner

The runner selects HVGs from raw counts, builds method-only metadata without `celltype`, trains scDisInFact independently for each seed, extracts `mu_c`, then restores the exact original Dataset 0 row order. The reordering is required because `create_scdisinfact_dataset()` splits cells into batch-condition matrices.


In [ ]:
from pathlib import Path
import gc, json, random
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch

from scDisInFact import create_scdisinfact_dataset, scdisinfact
from scrarebench import MethodOutput

_BARCODE_COL = "_scrarebench_original_barcode"

def _looks_like_raw_counts(matrix, max_values=100_000, seed=42):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()
    values = np.asarray(values)
    if len(values) > max_values:
        rng = np.random.default_rng(int(seed))
        values = values[rng.choice(len(values), size=max_values, replace=False)]
    values = values.astype(float, copy=False)
    if len(values) == 0:
        return True
    return bool(np.isfinite(values).all() and values.min() >= 0 and np.allclose(values, np.rint(values), atol=1e-6))


def run_scdisinfact(method_adata, seed, config):
    batch_key = info["batch_key"]
    count_layer = info.get("count_layer") or "counts"
    original_order = method_adata.obs_names.astype(str).to_numpy(copy=True)

    if "feature_types" in method_adata.var.columns:
        ft = method_adata.var["feature_types"].astype(str).str.upper()
        rna_mask = ft.eq("GEX").to_numpy()
        if rna_mask.sum() == 0:
            raise ValueError("No GEX features found.")
        if rna_mask.sum() != method_adata.n_vars:
            method_adata = method_adata[:, rna_mask].copy()
    method_adata.var_names_make_unique()

    if count_layer not in method_adata.layers:
        raise KeyError(f"Required raw-count layer {count_layer!r} is missing.")
    if not _looks_like_raw_counts(method_adata.layers[count_layer], seed=seed):
        raise ValueError(f"adata.layers[{count_layer!r}] does not look like raw counts.")

    sc.pp.highly_variable_genes(
        method_adata,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(int(config["n_hvg"]), method_adata.n_vars),
        batch_key=batch_key,
        span=float(config["hvg_span"]),
        subset=False,
        check_values=True,
    )
    hvg_mask = method_adata.var["highly_variable"].fillna(False).to_numpy()
    n_selected = int(hvg_mask.sum())
    if n_selected == 0:
        raise RuntimeError("No HVGs selected.")

    counts_hvg = method_adata.layers[count_layer][:, hvg_mask]
    counts_hvg = counts_hvg.tocsr() if sp.issparse(counts_hvg) else np.asarray(counts_hvg)
    cell_sums = np.asarray(counts_hvg.sum(axis=1)).ravel()
    if np.any(cell_sums <= 0):
        raise ValueError(f"{int(np.sum(cell_sums <= 0))} cells have zero counts across selected HVGs.")

    condition_col = str(config["condition_column"])
    meta_cells = pd.DataFrame(index=method_adata.obs_names.copy())
    meta_cells[batch_key] = method_adata.obs[batch_key].astype(str).to_numpy()
    meta_cells[condition_col] = str(config["condition_value"])
    meta_cells[_BARCODE_COL] = original_order

    random.seed(int(seed)); np.random.seed(int(seed)); torch.manual_seed(int(seed))
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(int(seed))
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    seed_dir = Path(METHOD_WORK_DIR) / f"seed_{int(seed)}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    print(f"Seed {seed}: {method_adata.n_obs:,} cells x {n_selected:,} HVGs; device={device}")
    print("Condition policy: one constant method-only condition; benchmark labels are not used.")

    data_dict = create_scdisinfact_dataset(
        counts_hvg,
        meta_cells,
        condition_key=[condition_col],
        batch_key=batch_key,
        batch_cond_key=None,
        log_trans=True,
    )

    model = scdisinfact(
        data_dict=data_dict,
        reg_mmd_comm=float(config["reg_mmd_comm"]),
        reg_mmd_diff=float(config["reg_mmd_diff"]),
        reg_gl=float(config["reg_gl"]),
        reg_class=float(config["reg_class"]),
        reg_kl_comm=float(config["reg_kl_comm"]),
        reg_kl_diff=float(config["reg_kl_diff"]),
        Ks=list(config["Ks"]),
        batch_size=int(config["batch_size"]),
        interval=int(config["interval"]),
        lr=float(config["lr"]),
        seed=int(seed),
        enc_injection=bool(config["enc_injection"]),
        device=device,
    )

    losses = model.train_model(nepochs=int(config["nepochs"]), recon_loss=str(config["recon_loss"]))
    model.eval()

    latent_parts, barcode_parts = [], []
    with torch.no_grad():
        for dataset, meta_part in zip(data_dict["datasets"], data_dict["meta_cells"]):
            inf = model.inference(
                counts=dataset.counts_norm.to(model.device),
                batch_ids=dataset.batch_id[:, None].to(model.device),
                print_stat=False,
            )
            latent_parts.append(inf["mu_c"].detach().cpu().numpy().astype(np.float32, copy=False))
            barcode_parts.append(meta_part[_BARCODE_COL].astype(str).to_numpy())

    latent_native = np.concatenate(latent_parts, axis=0)
    native_barcodes = np.concatenate(barcode_parts, axis=0)

    if latent_native.shape != (method_adata.n_obs, int(config["Ks"][0])):
        raise RuntimeError(f"Unexpected shared latent shape: {latent_native.shape}")
    if not np.isfinite(latent_native).all():
        raise ValueError("Shared latent contains NaN/Inf.")
    if len(np.unique(native_barcodes)) != len(native_barcodes):
        raise RuntimeError("Recovered barcodes are not unique.")
    if set(native_barcodes) != set(original_order):
        raise RuntimeError("scDisInFact output cell set differs from Dataset 0.")

    pos = pd.Series(np.arange(len(native_barcodes)), index=native_barcodes)
    reorder = pos.loc[original_order].to_numpy()
    latent = latent_native[reorder]
    assert np.array_equal(native_barcodes[reorder], original_order)

    loss_names = ["total", "reconstruction", "kl_common", "kl_diff", "mmd_common", "mmd_diff", "classification", "group_lasso_diff"]
    loss_df = pd.DataFrame({name: vals for name, vals in zip(loss_names, losses)})
    loss_path = seed_dir / "training_validation_losses.csv"
    loss_df.to_csv(loss_path, index=False)

    config_path = seed_dir / "method_config.json"
    payload = dict(config)
    payload.update({"seed": int(seed), "selected_hvgs": n_selected, "upstream_commit": SCDISINFACT_COMMIT,
                    "condition_policy": "constant baseline; evaluate shared mu_c"})
    config_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    print(f"Seed {seed}: aligned shared latent {latent.shape}")

    del model, data_dict, latent_native, counts_hvg, pos, reorder
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=original_order,
        representation_key="X_scdisinfact_shared",
        provenance_files={
            "scdisinfact_training_losses": loss_path,
            "scdisinfact_method_config": config_path,
        },
    )


## 5. Run the High-Level multi-seed benchmark

For a quick compatibility check, temporarily use `METHOD_SEEDS = [42]`. Restore all three seeds for the final run.


In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scDisInFact",
    runner=run_scdisinfact,
    config=METHOD_CONFIG,
    dependencies=METHOD_RUNTIME_DEPENDENCIES + (SCDISINFACT_GITHUB,),
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        "include_latent_in_bundle": False,
    },
    install_dependencies=False,
    finalize=True,
)


## 6. Inspect the multi-seed summary


In [ ]:
from IPython.display import display

display(result.summary().round(5))
print("Method:", result.method)
print("Completed seeds:", result.seeds)
print("Multi-seed report:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(f"\nSeed {seed}")
    print("  representation:", run.representation_key)
    print("  output:", run.output_dir)
    print("  interactive report:", run.interactive_report_path)
    print("  PDF:", run.pdf_path)
    print("  bundle:", run.bundle_path)


## 7. Download the final multi-seed archive


In [ ]:
from pathlib import Path
from google.colab import files

archive_path = Path(result.archive_path)
if not archive_path.exists():
    raise FileNotFoundError(f"Output archive not found: {archive_path}")
print(f"Downloading: {archive_path.name} ({archive_path.stat().st_size/(1024**2):.2f} MB)")
files.download(str(archive_path))
